이 자료는 2021년 12월 14일에 마지막으로 테스트되었습니다.

이 자료는 위키독스 '딥 러닝을 이용한 자연어 처리 입문'에서 BLEU 구현하기 튜토리얼입니다.  
링크 : https://wikidocs.net/31695

- 일반적인 언어 모델의 성능은 [퍼플렉시티](../10.%20Introduction%20to%20Language%20Model/10-03.%20perplexity.ipynb)로 측정함
- 이는 생성된 문장의 품질은 평가할 수 있으나, 번역의 품질을 측정하기에는 부적합
- 기계번역의 품질을 평가하는 지표로는 BLEU(Bilingual Evaluation Understudy)이 대표적임

In [1]:
import numpy as np
from collections import Counter
from nltk import ngrams

# 1. BLEU(Bilingual Evaluation Understudy)

$$\text{BLEU} = \text{BP} × \exp(\sum_{n=1}^{N}w_{n}\ \ln p_{n})$$

- 기계 번역 결과와 사람이 직접 번역한 결과가 얼마나 유사한지 n-gram에 기반하여 비교
- 장점
    - 언어에 구애받지 않고 사용 가능
    - 계산 속도가 빠름
- 단점
    - 사람의 번역이 필요하여 평가과정을 완전히 객관화 및 자동화할 수 없음

## 1.1 단어 개수 카운트
### 1.11 유니그램 정밀도(Unigram Precision)
- 비교 대상 번역기가 번역한 문장을 `Candiate #`, 사람이 번역한 평가 기준을 `Reference #`이라 함
- `Candidate #`에서 등장한 단어 중 `Reference #`에서 한 번이라도 등장한 것의 숫자를 셈

### 1.12 보정된 유니그램 정밀도(Modified Unigram Precision)
- 유니그램 정밀도는 번역문에 단어가 중복해서 등장할 경우 정확도가 과도하게 측정될 수 있음
- 이를 보정하기 위해, 유니그램이 각각의 `Reference #`에서 등장한 최대 횟수로 제한함

In [2]:
# 토큰화 된 문장(tokens)에서 n-gram을 카운트
def simple_count(tokens, n):
  return Counter(ngrams(tokens, n))

In [3]:
candidate = "It is a guide to action which ensures that the military always obeys the commands of the party."
tokens = candidate.split() # 토큰화
result = simple_count(tokens, 1) # n = 1은 유니그램
print('유니그램 카운트 :',result)

유니그램 카운트 : Counter({('the',): 3, ('It',): 1, ('is',): 1, ('a',): 1, ('guide',): 1, ('to',): 1, ('action',): 1, ('which',): 1, ('ensures',): 1, ('that',): 1, ('military',): 1, ('always',): 1, ('obeys',): 1, ('commands',): 1, ('of',): 1, ('party.',): 1})


In [4]:
candidate = 'the the the the the the the'
tokens = candidate.split() # 토큰화
result = simple_count(tokens, 1) # n = 1은 유니그램
print('유니그램 카운트 :',result)

유니그램 카운트 : Counter({('the',): 7})


In [5]:
def count_clip(candidate, reference_list, n):
  # Ca 문장에서 n-gram 카운트
  ca_cnt = simple_count(candidate, n)
  max_ref_cnt_dict = dict()

  for ref in reference_list:
    # Ref 문장에서 n-gram 카운트
    ref_cnt = simple_count(ref, n)

    # 각 Ref 문장에 대해서 비교하여 n-gram의 최대 등장 횟수를 계산.
    for n_gram in ref_cnt:
      if n_gram in max_ref_cnt_dict:
        max_ref_cnt_dict[n_gram] = max(ref_cnt[n_gram], max_ref_cnt_dict[n_gram])
      else:
        max_ref_cnt_dict[n_gram] = ref_cnt[n_gram]

  return {
        # count_clip = min(count, max_ref_count)
        n_gram: min(ca_cnt.get(n_gram, 0), max_ref_cnt_dict.get(n_gram, 0)) for n_gram in ca_cnt
     }

In [6]:
candidate = 'the the the the the the the'
references = [
    'the cat is on the mat',
    'there is a cat on the mat'
]
result = count_clip(candidate.split(),list(map(lambda ref: ref.split(), references)),1)
print('보정된 유니그램 카운트 :',result)

보정된 유니그램 카운트 : {('the',): 2}


In [7]:
def modified_precision(candidate, reference_list, n):
  clip_cnt = count_clip(candidate, reference_list, n)
  total_clip_cnt = sum(clip_cnt.values()) # 분자

  cnt = simple_count(candidate, n)
  total_cnt = sum(cnt.values()) # 분모

  # 분모가 0이 되는 것을 방지
  if total_cnt == 0:
    total_cnt = 1

  # 분자 : count_clip의 합, 분모 : 단순 count의 합 ==> 보정된 정밀도
  return (total_clip_cnt / total_cnt)

In [8]:
result = modified_precision(candidate.split(), list(map(lambda ref: ref.split(), references)), n=1)
print('보정된 유니그램 정밀도 :',result)

보정된 유니그램 정밀도 : 0.2857142857142857


### 1.13 n-gram 정밀도
- 유니그램 정밀도는 단어의 순서를 고려하지 않는 한계가 있음
- n개 단어의 순서를 고려하기 위해서는 n-gram 정밀도로 확장할 수 있음
- 이를 더욱 확장하여, 정의된 모든 n에 대한 정밀도를 아래와 같이 조합할 수 있음

$$\frac{\text{BLEU}}{\text{BP}} = exp(\sum_{n=1}^{N}w_{n}\ \text{log}\ p_{n})$$

## 1.2 짧음 제제(Brevity Penalty)
- n-gram 정밀도는 `Candidate #`의 토큰 수를 분모로 삼으므로, 길이가 짧을 수록 유리함
- 따라서, 해당하는 `Reference #`의 길이보다 짧은 Candidate의 정밀도를 낮춰 보정함

$$\text{BP} = \begin{cases}1&\text{if}\space c>r\\ e^{(1-r/c)}&\text{if}\space c \leq r \end{cases}$$
$c$ : Candidate의 길이 </br>
$r$ : Candidate와 가장 길이 차이가 작은 Reference의 길이

In [9]:
# Ca 길이와 가장 근접한 Ref의 길이를 리턴하는 함수
def closest_ref_length(candidate, reference_list):
  ca_len = len(candidate) # ca 길이
  ref_lens = (len(ref) for ref in reference_list) # Ref들의 길이
  # 길이 차이를 최소화하는 Ref를 찾아서 Ref의 길이를 리턴
  closest_ref_len = min(ref_lens, key=lambda ref_len: (abs(ref_len - ca_len), ref_len))
  return closest_ref_len

In [10]:
def brevity_penalty(candidate, reference_list):
  ca_len = len(candidate)
  ref_len = closest_ref_length(candidate, reference_list)

  if ca_len > ref_len:
    return 1

  # candidate가 비어있다면 BP = 0 → BLEU = 0.0
  elif ca_len == 0 :
    return 0
  else:
    return np.exp(1 - ref_len/ca_len)

In [11]:
def bleu_score(candidate, reference_list, weights=[0.25, 0.25, 0.25, 0.25]):
  bp = brevity_penalty(candidate, reference_list) # 브레버티 패널티, BP

  p_n = [modified_precision(candidate, reference_list, n=n) for n, _ in enumerate(weights,start=1)]
  # p1, p2, p3, ..., pn
  score = np.sum([w_i * np.log(p_i) if p_i != 0 else 0 for w_i, p_i in zip(weights, p_n)])
  return bp * np.exp(score)

In [12]:
import nltk.translate.bleu_score as bleu

candidate = 'It is a guide to action which ensures that the military always obeys the commands of the party'
references = [
    'It is a guide to action that ensures that the military will forever heed Party commands',
    'It is the guiding principle which guarantees the military forces always being under the command of the Party',
    'It is the practical guide for the army always to heed the directions of the party'
]

print('실습 코드의 BLEU :',bleu_score(candidate.split(),list(map(lambda ref: ref.split(), references))))
print('패키지 NLTK의 BLEU :',bleu.sentence_bleu(list(map(lambda ref: ref.split(), references)),candidate.split()))

실습 코드의 BLEU : 0.5045666840058485
패키지 NLTK의 BLEU : 0.5045666840058485
